In [1]:
import hashlib
import json
import math
import urllib.request
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
import csv
import numpy as np
import contextlib
import io
from IPython.display import HTML, display


In [2]:
PATTERN_COUNT = 243
GREEN_ID = 242  # (2, 2, 2, 2, 2) in base 3


def load_words_from_url(url):
    with urllib.request.urlopen(url) as response:
        lines = response.read().decode("utf-8").splitlines()

    return sorted({
        word.strip().lower()
        for word in lines
        if len(word.strip()) == 5 and word.strip().isalpha()
    })


def get_pattern_tuple(answer, guess):
    """
    Wordle feedback:
        0 = gray
        1 = yellow
        2 = green

    Duplicate letters are handled by consuming greens first,
    then assigning yellows from the remaining answer letters.
    """
    pattern = [0] * 5
    remaining = Counter()

    for i in range(5):
        if guess[i] == answer[i]:
            pattern[i] = 2
        else:
            remaining[answer[i]] += 1

    for i in range(5):
        if pattern[i] == 2:
            continue

        letter = guess[i]
        if remaining[letter] > 0:
            pattern[i] = 1
            remaining[letter] -= 1

    return tuple(pattern)


def encode_pattern(pattern):
    return sum(value * (3 ** i) for i, value in enumerate(pattern))


def build_pattern_table(answers, guesses, progress_every=100):
    """
    table[answer_index, guess_index] = encoded Wordle response.
    """
    table = np.empty(
        (len(answers), len(guesses)),
        dtype=np.uint8,
    )

    for answer_index, answer in enumerate(answers):
        for guess_index, guess in enumerate(guesses):
            table[answer_index, guess_index] = encode_pattern(
                get_pattern_tuple(answer, guess)
            )

        completed = answer_index + 1
        if completed % progress_every == 0 or completed == len(answers):
            print(
                f"pattern table: {completed}/{len(answers)} answers"
            )

    return table


def load_or_build_pattern_table(
    answers,
    guesses,
    cache_path,
):
    """
    Reuse the repository-compatible cache when its shape is correct.
    """
    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    expected_shape = (len(answers), len(guesses))

    if cache_path.exists():
        cached = np.load(cache_path, mmap_mode="r")

        if cached.shape == expected_shape and cached.dtype == np.uint8:
            print("loaded cached pattern table:", cache_path)
            return cached

        print(
            "cached pattern table has the wrong shape or dtype; rebuilding"
        )

    print("building pattern table; the first run is the slow run")
    table = build_pattern_table(answers, guesses)
    np.save(cache_path, table)
    print("saved pattern table:", cache_path)

    return np.load(cache_path, mmap_mode="r")


@dataclass(frozen=True)
class Decision:
    """
    Bellman value and selected action for one candidate set.

    mean_attempts:
        Expected number of additional guesses from this state.

    max_attempts:
        Worst-case number of additional guesses from this state.
    """
    mean_attempts: float
    max_attempts: int
    chosen_lambda: float | None
    optimal_lambdas: tuple
    guess_index: int


class BellmanDynamicMEM:
    """
    Statewise dynamic MEM.

    At candidate set C:

    1. For every lambda in lambda_grid, MEM chooses one current guess
       g_lambda(C).

    2. That guess is evaluated by

           Q(C, lambda)
             = 1 + sum_p P(p | C, g_lambda) V(C_p),

       excluding the green branch from the future-cost sum.

    3. Every child value V(C_p) is itself computed by searching the
       lambda grid again:

           V(C_p) = min_lambda Q(C_p, lambda).

    Thus lambda controls only the current guess.  It is not held fixed
    on later turns.
    """

    def __init__(
        self,
        answers,
        guesses,
        pattern_table,
        lambda_grid,
        tie_tolerance=1e-12,
        profile_progress_every=100,
        solve_progress_every=100,
        small_state_pairwise_threshold=15,
    ):
        self.answers = answers
        self.guesses = guesses
        self.pattern_table = pattern_table
        self.lambda_grid = tuple(
            sorted({round(float(value), 10) for value in lambda_grid})
        )

        if not self.lambda_grid:
            raise ValueError("lambda_grid must not be empty.")

        if self.lambda_grid[0] < 0 or self.lambda_grid[-1] > 1:
            raise ValueError("Every lambda must be between 0 and 1.")

        self.tie_tolerance = float(tie_tolerance)
        self.profile_progress_every = int(profile_progress_every)
        self.solve_progress_every = int(solve_progress_every)
        self.small_state_pairwise_threshold = max(
            2,
            int(small_state_pairwise_threshold),
        )

        self.guess_to_index = {
            word: index
            for index, word in enumerate(guesses)
        }
        self.answer_to_index = {
            word: index
            for index, word in enumerate(answers)
        }
        self.answer_to_guess_index = np.array(
            [self.guess_to_index[word] for word in answers],
            dtype=np.int32,
        )

        self.all_guess_indices = np.arange(
            len(guesses),
            dtype=np.int32,
        )

        # Caches
        self.best_guess_by_lambda_cache = {}
        self.partition_cache = {}
        self.guess_value_cache = {}
        self.decision_cache = {}
        self.lambda_rows_cache = {}

        self.profiled_candidate_sets = 0
        self.solved_dynamic_states = 0
        self._active_states = set()

    @staticmethod
    def _state_array(state):
        return np.fromiter(
            state,
            dtype=np.int32,
            count=len(state),
        )

    def _partition_state(self, state, guess_index):
        """
        Return ((pattern_id, child_state), ...) for a state and guess.
        """
        key = (state, int(guess_index))

        cached = self.partition_cache.get(key)
        if cached is not None:
            return cached

        candidate_array = self._state_array(state)
        patterns = self.pattern_table[
            candidate_array,
            int(guess_index),
        ]

        groups = defaultdict(list)

        for answer_index, pattern_id in zip(state, patterns):
            groups[int(pattern_id)].append(int(answer_index))

        result = tuple(
            (pattern_id, tuple(indices))
            for pattern_id, indices in sorted(groups.items())
        )

        self.partition_cache[key] = result
        return result

    @staticmethod
    def _select_best_guess(
        log_score,
        expected_remaining,
        max_block,
        candidate_guess_mask,
    ):
        """Select the lexicographic winner without sorting all guesses."""
        candidates = np.flatnonzero(
            log_score == np.max(log_score)
        )

        best_expected = np.min(
            expected_remaining[candidates]
        )
        candidates = candidates[
            expected_remaining[candidates] == best_expected
        ]

        best_max_block = np.min(max_block[candidates])
        candidates = candidates[
            max_block[candidates] == best_max_block
        ]

        prefer_candidate = np.max(
            candidate_guess_mask[candidates]
        )
        candidates = candidates[
            candidate_guess_mask[candidates] == prefer_candidate
        ]

        # guesses is sorted, so the smallest index is lexical-first.
        return int(np.min(candidates))

    def _profile_state(self, state):
        """
        Find the MEM-best current guess for every lambda.

        Two implementation paths are used:

        * Small states compute block sizes from pairwise pattern equality.
          This avoids allocating a fixed (n_guesses, 243) matrix for the
          many states containing only a few candidates.

        * Larger states use the original pattern-count matrix.

        Lambda winners are selected by linear filtering rather than a full
        lexicographic sort of all guesses for every lambda.
        """
        cached = self.best_guess_by_lambda_cache.get(state)
        if cached is not None:
            return cached

        n_candidates = len(state)

        if n_candidates <= 1:
            raise ValueError(
                "_profile_state must only be called for states of size > 1."
            )

        candidate_array = self._state_array(state)
        n_guesses = len(self.guesses)

        # Shape: (n_guesses, n_candidates).
        response_matrix = np.ascontiguousarray(
            self.pattern_table[candidate_array, :].T
        )

        if n_candidates <= self.small_state_pairwise_threshold:
            # equal_counts[g, j] is the size of the feedback block
            # containing candidate j for guess g.  Therefore:
            #   sum_j equal_counts[g, j] = sum_blocks block_size^2
            # and
            #   sum_j 1/equal_counts[g, j] = number of blocks.
            equal_counts = (
                response_matrix[:, :, None]
                == response_matrix[:, None, :]
            ).sum(axis=2)

            expected_remaining = (
                equal_counts.sum(axis=1, dtype=np.float64)
                / n_candidates
            )
            max_block = equal_counts.max(axis=1).astype(
                np.int32
            )
            block_count = np.rint(
                (1.0 / equal_counts).sum(axis=1)
            ).astype(np.int16)
            entropy = (
                math.log2(n_candidates)
                - np.log2(equal_counts).sum(axis=1)
                / n_candidates
            )
        else:
            counts = np.zeros(
                (n_guesses, PATTERN_COUNT),
                dtype=np.uint16,
            )
            row_indices = self.all_guess_indices

            for column in range(n_candidates):
                counts[
                    row_indices,
                    response_matrix[:, column],
                ] += 1

            block_count = (counts > 0).sum(
                axis=1
            ).astype(np.int16)
            max_block = counts.max(axis=1).astype(np.int32)

            # Avoid materializing a full float64 copy of counts.
            expected_remaining = np.square(
                counts,
                dtype=np.float64,
            ).sum(axis=1) / n_candidates

            c_log_c = np.zeros(
                n_candidates + 1,
                dtype=np.float64,
            )
            positive_counts = np.arange(
                1,
                n_candidates + 1,
                dtype=np.float64,
            )
            c_log_c[1:] = (
                positive_counts
                * np.log2(positive_counts)
            )
            entropy = (
                math.log2(n_candidates)
                - c_log_c[counts].sum(axis=1)
                / n_candidates
            )

        max_possible_blocks = min(
            PATTERN_COUNT,
            n_candidates,
        )
        log_max_blocks = math.log2(max_possible_blocks)

        block_score = np.zeros(
            n_guesses,
            dtype=np.float64,
        )
        uniformity_score = np.zeros(
            n_guesses,
            dtype=np.float64,
        )

        valid = block_count > 1
        block_score[valid] = (
            np.log2(block_count[valid])
            / log_max_blocks
        )
        uniformity_score[valid] = (
            entropy[valid]
            / np.log2(block_count[valid])
        )

        candidate_guess_mask = np.zeros(
            n_guesses,
            dtype=np.int8,
        )
        candidate_guess_mask[
            self.answer_to_guess_index[candidate_array]
        ] = 1

        positive_components = (
            (block_score > 0)
            & (uniformity_score > 0)
        )
        log_block_score = np.zeros(
            n_guesses,
            dtype=np.float64,
        )
        log_uniformity_score = np.zeros(
            n_guesses,
            dtype=np.float64,
        )
        log_block_score[positive_components] = np.log(
            block_score[positive_components]
        )
        log_uniformity_score[positive_components] = np.log(
            uniformity_score[positive_components]
        )

        best_by_lambda = {}

        for lam in self.lambda_grid:
            score = np.zeros(
                n_guesses,
                dtype=np.float64,
            )
            score[positive_components] = np.exp(
                2.0 * lam
                * log_block_score[positive_components]
                + 2.0 * (1.0 - lam)
                * log_uniformity_score[positive_components]
            )

            best_by_lambda[lam] = self._select_best_guess(
                log_score=score,
                expected_remaining=expected_remaining,
                max_block=max_block,
                candidate_guess_mask=candidate_guess_mask,
            )

        self.best_guess_by_lambda_cache[state] = best_by_lambda
        self.profiled_candidate_sets += 1

        if (
            self.profile_progress_every > 0
            and (
                self.profiled_candidate_sets == 1
                or self.profiled_candidate_sets
                % self.profile_progress_every
                == 0
            )
        ):
            print(
                "profiled candidate sets:",
                self.profiled_candidate_sets,
                "latest size:",
                n_candidates,
            )

        return best_by_lambda

    def get_lambda_rows(self, state):
        """Return lambda rows, generating the two-state case lazily."""
        state = tuple(state)
        cached = self.lambda_rows_cache.get(state)
        if cached is not None:
            return cached

        if len(state) != 2:
            raise KeyError(
                "Lambda rows are unavailable for an unsolved state."
            )

        decision = self.solve_state(state)
        return tuple({
            "lambda": lam,
            "guess_index": decision.guess_index,
            "guess": self.guesses[decision.guess_index],
            "mean_attempts": decision.mean_attempts,
            "max_attempts": decision.max_attempts,
        } for lam in self.lambda_grid)

    def _evaluate_current_guess(self, state, guess_index):
        """
        Bellman value of using guess_index now and dynamically
        re-optimizing lambda in every non-green child state.
        """
        key = (state, int(guess_index))

        cached = self.guess_value_cache.get(key)
        if cached is not None:
            return cached

        n_candidates = len(state)
        weighted_future_mean = 0.0
        future_max = 0

        for pattern_id, child_state in self._partition_state(
            state,
            guess_index,
        ):
            if pattern_id == GREEN_ID:
                # The current guess already solves this branch.
                continue

            if child_state == state:
                # A non-progressing action would create a Bellman cycle.
                # It can never be preferable to a valid splitting guess.
                result = (math.inf, 10**9)
                self.guess_value_cache[key] = result
                return result

            child_decision = self.solve_state(child_state)

            weighted_future_mean += (
                len(child_state)
                / n_candidates
                * child_decision.mean_attempts
            )
            future_max = max(
                future_max,
                child_decision.max_attempts,
            )

        result = (
            1.0 + weighted_future_mean,
            1 + future_max,
        )
        self.guess_value_cache[key] = result
        return result

    def solve_state(self, state):
        """
        Return the dynamically optimal lambda decision for state.

        States of size two have a closed-form solution: guessing the
        lexical-first remaining answer always splits the two candidates,
        yielding mean 1.5 and maximum 2 additional guesses.
        """
        state = tuple(state)

        if len(state) == 0:
            raise ValueError("Candidate state must not be empty.")

        if len(state) == 1:
            answer_index = state[0]
            return Decision(
                mean_attempts=1.0,
                max_attempts=1,
                chosen_lambda=None,
                optimal_lambdas=(),
                guess_index=int(
                    self.answer_to_guess_index[answer_index]
                ),
            )

        cached = self.decision_cache.get(state)
        if cached is not None:
            return cached

        if len(state) == 2:
            candidate_array = self._state_array(state)
            guess_index = int(np.min(
                self.answer_to_guess_index[candidate_array]
            ))
            decision = Decision(
                mean_attempts=1.5,
                max_attempts=2,
                chosen_lambda=max(self.lambda_grid),
                optimal_lambdas=self.lambda_grid,
                guess_index=guess_index,
            )
            self.decision_cache[state] = decision
            self.solved_dynamic_states += 1

            if (
                self.solve_progress_every > 0
                and self.solved_dynamic_states
                % self.solve_progress_every
                == 0
            ):
                print(
                    "dynamic Bellman states:",
                    self.solved_dynamic_states,
                )

            return decision

        if state in self._active_states:
            raise RuntimeError(
                "Unexpected Bellman cycle for candidate state."
            )

        self._active_states.add(state)

        try:
            best_guess_by_lambda = self._profile_state(state)

            # Different lambdas frequently choose exactly the same guess.
            # Evaluate each distinct action only once.
            unique_guess_values = {}

            for guess_index in set(best_guess_by_lambda.values()):
                unique_guess_values[guess_index] = (
                    self._evaluate_current_guess(
                        state,
                        guess_index,
                    )
                )

            rows = []

            for lam in self.lambda_grid:
                guess_index = best_guess_by_lambda[lam]
                mean_attempts, max_attempts = (
                    unique_guess_values[guess_index]
                )

                rows.append({
                    "lambda": lam,
                    "guess_index": guess_index,
                    "guess": self.guesses[guess_index],
                    "mean_attempts": mean_attempts,
                    "max_attempts": max_attempts,
                })

            finite_rows = [
                row
                for row in rows
                if math.isfinite(row["mean_attempts"])
            ]

            if not finite_rows:
                raise RuntimeError(
                    "No progressing MEM action was found."
                )

            best_mean = min(
                row["mean_attempts"]
                for row in finite_rows
            )

            mean_tied = [
                row
                for row in finite_rows
                if abs(
                    row["mean_attempts"] - best_mean
                ) <= self.tie_tolerance
            ]

            best_max = min(
                row["max_attempts"]
                for row in mean_tied
            )

            optimal_rows = [
                row
                for row in mean_tied
                if row["max_attempts"] == best_max
            ]

            optimal_lambdas = tuple(
                row["lambda"]
                for row in optimal_rows
            )
            chosen_lambda = max(optimal_lambdas)
            chosen_row = next(
                row
                for row in optimal_rows
                if row["lambda"] == chosen_lambda
            )

            decision = Decision(
                mean_attempts=chosen_row["mean_attempts"],
                max_attempts=chosen_row["max_attempts"],
                chosen_lambda=chosen_lambda,
                optimal_lambdas=optimal_lambdas,
                guess_index=chosen_row["guess_index"],
            )

            self.lambda_rows_cache[state] = tuple(rows)
            self.decision_cache[state] = decision
            self.solved_dynamic_states += 1

            if (
                self.solve_progress_every > 0
                and self.solved_dynamic_states
                % self.solve_progress_every
                == 0
            ):
                print(
                    "dynamic Bellman states:",
                    self.solved_dynamic_states,
                )

            return decision

        finally:
            self._active_states.remove(state)

    def forced_root_decision(self, root_state, root_lambda):
        """
        Force the first-turn lambda, but evaluate all later states using
        the Bellman dynamic policy.
        """
        root_state = tuple(root_state)
        root_lambda = round(float(root_lambda), 10)

        if root_lambda not in self.lambda_grid:
            raise ValueError(
                "ROOT_LAMBDA must be included in LAMBDA_GRID."
            )

        best_guess_by_lambda = self._profile_state(root_state)
        guess_index = best_guess_by_lambda[root_lambda]

        mean_attempts, max_attempts = (
            self._evaluate_current_guess(
                root_state,
                guess_index,
            )
        )

        return Decision(
            mean_attempts=mean_attempts,
            max_attempts=max_attempts,
            chosen_lambda=root_lambda,
            optimal_lambdas=(root_lambda,),
            guess_index=guess_index,
        )

    def _next_state_for_answer(
        self,
        state,
        guess_index,
        answer_index,
    ):
        pattern_id = int(
            self.pattern_table[
                answer_index,
                guess_index,
            ]
        )

        if pattern_id == GREEN_ID:
            return pattern_id, None

        for child_pattern, child_state in self._partition_state(
            state,
            guess_index,
        ):
            if child_pattern == pattern_id:
                return pattern_id, child_state

        raise RuntimeError(
            "The observed answer pattern has no child state."
        )

    def evaluate_full_policy(
        self,
        root_state,
        root_decision,
    ):
        """
        Traverse the final selected policy for every possible answer.
        """
        root_state = tuple(root_state)

        attempts_by_answer = {}
        flows_by_answer = {}

        reachable_states = {}
        minimum_depth_by_state = {}

        for answer_index, answer in enumerate(self.answers):
            state = root_state
            depth = 0
            flow = []

            while True:
                if len(state) == 1:
                    decision = self.solve_state(state)
                elif state == root_state:
                    decision = root_decision
                else:
                    decision = self.solve_state(state)

                minimum_depth_by_state[state] = min(
                    depth,
                    minimum_depth_by_state.get(
                        state,
                        depth,
                    ),
                )
                reachable_states[state] = decision

                flow.append({
                    "state": state,
                    "n_candidates": len(state),
                    "guess_index": decision.guess_index,
                    "guess": self.guesses[
                        decision.guess_index
                    ],
                    "lambda": decision.chosen_lambda,
                    "optimal_lambdas": (
                        decision.optimal_lambdas
                    ),
                    "bellman_mean": (
                        decision.mean_attempts
                    ),
                    "bellman_max": (
                        decision.max_attempts
                    ),
                })

                pattern_id, next_state = (
                    self._next_state_for_answer(
                        state,
                        decision.guess_index,
                        answer_index,
                    )
                )

                if pattern_id == GREEN_ID:
                    break

                state = next_state
                depth += 1

            attempts_by_answer[answer] = len(flow)
            flows_by_answer[answer] = flow

        return {
            "attempts_by_answer": attempts_by_answer,
            "flows_by_answer": flows_by_answer,
            "reachable_states": reachable_states,
            "minimum_depth_by_state": minimum_depth_by_state,
        }

    @staticmethod
    def format_flow(flow):
        pieces = []

        for step in flow:
            if step["lambda"] is None:
                pieces.append(
                    f'{step["guess"]} '
                    f'[solve, n={step["n_candidates"]}]'
                )
                continue

            optimal = step["optimal_lambdas"]

            if len(optimal) <= 1:
                optimal_text = ""
            else:
                values = ",".join(
                    f"{value:.2f}"
                    for value in optimal
                )
                optimal_text = f", optimal={{{values}}}"

            pieces.append(
                f'{step["guess"]} '
                f'[lambda={step["lambda"]:.2f}'
                f'{optimal_text}, '
                f'n={step["n_candidates"]}]'
            )

        return " -> ".join(pieces)

    def print_answer_detail(
        self,
        answer,
        policy_result,
        root_state,
        root_decision,
    ):
        """
        Print the lambda comparison at every state on one answer path.
        """
        if answer not in policy_result["flows_by_answer"]:
            raise ValueError(f"Unknown answer: {answer}")

        print()
        print(
            answer + ":",
            self.format_flow(
                policy_result["flows_by_answer"][answer]
            ),
        )

        for step_number, step in enumerate(
            policy_result["flows_by_answer"][answer],
            start=1,
        ):
            state = step["state"]

            print()
            print(
                f"step {step_number}: "
                f'n={step["n_candidates"]}, '
                f'guess={step["guess"]}'
            )

            if len(state) == 1:
                print("  only one candidate remains; guess it")
                continue

            rows = self.get_lambda_rows(state)
            decision = self.decision_cache[state]

            for row in rows:
                if row["lambda"] == decision.chosen_lambda:
                    marker = "*"
                elif row["lambda"] in decision.optimal_lambdas:
                    marker = "="
                else:
                    marker = " "

                print(
                    f'{marker} lambda={row["lambda"]:.2f} '
                    f'guess={row["guess"]:<5} '
                    f'bellman_mean='
                    f'{row["mean_attempts"]:.6f} '
                    f'bellman_max='
                    f'{row["max_attempts"]}'
                )


def summarize_policy(
    solver,
    policy_result,
    root_decision,
):
    attempts = policy_result["attempts_by_answer"]
    values = np.array(
        list(attempts.values()),
        dtype=np.int32,
    )

    distribution = Counter(values.tolist())
    max_attempts = int(values.max())

    worst_answers = sorted(
        answer
        for answer, count in attempts.items()
        if count == max_attempts
    )

    representative_state_counts = Counter()
    unique_optimum_counts = Counter()
    optimum_inclusion_counts = Counter()
    lambda_depth_counts = defaultdict(Counter)
    answer_weighted_counts = Counter()
    tied_state_count = 0

    for state, decision in policy_result[
        "reachable_states"
    ].items():
        if decision.chosen_lambda is None:
            continue

        representative_state_counts[
            decision.chosen_lambda
        ] += 1

        depth = policy_result[
            "minimum_depth_by_state"
        ][state]
        lambda_depth_counts[depth][
            decision.chosen_lambda
        ] += 1

        if len(decision.optimal_lambdas) == 1:
            unique_optimum_counts[
                decision.optimal_lambdas[0]
            ] += 1
        else:
            tied_state_count += 1

        for lam in decision.optimal_lambdas:
            optimum_inclusion_counts[lam] += 1

    for flow in policy_result[
        "flows_by_answer"
    ].values():
        for step in flow:
            if step["lambda"] is not None:
                answer_weighted_counts[
                    step["lambda"]
                ] += 1

    print()
    print(
        "algorithm:",
        "Bellman Statewise Dynamic Modified Entropy Maximizing",
    )
    print(
        "lambda evaluation:",
        "current MEM guess + dynamically re-optimized child states",
    )
    print(
        "first guess:",
        solver.guesses[root_decision.guess_index],
    )
    print(
        "root lambda:",
        root_decision.chosen_lambda,
    )
    print(
        "root optimal lambdas:",
        root_decision.optimal_lambdas,
    )
    print(
        "mean attempts:",
        float(values.mean()),
    )
    print(
        "max attempts:",
        max_attempts,
    )
    print(
        "reachable policy states:",
        len(policy_result["reachable_states"]),
    )
    print(
        "searched dynamic states:",
        solver.solved_dynamic_states,
    )
    print(
        "profiled candidate sets:",
        solver.profiled_candidate_sets,
    )
    print(
        "evaluated state/guess actions:",
        len(solver.guess_value_cache),
    )
    print(
        "representative lambda state counts:",
        dict(sorted(representative_state_counts.items())),
    )
    print(
        "unique-optimum lambda counts:",
        dict(sorted(unique_optimum_counts.items())),
    )
    print(
        "states with multiple optimal lambdas:",
        tied_state_count,
    )
    print(
        "optimal-lambda inclusion counts:",
        dict(sorted(optimum_inclusion_counts.items())),
    )
    print(
        "answer-weighted representative lambda counts:",
        dict(sorted(answer_weighted_counts.items())),
    )
    print(
        "lambda depth counts:",
        {
            depth: dict(sorted(counts.items()))
            for depth, counts
            in sorted(lambda_depth_counts.items())
        },
    )

    print("attempt distribution:")

    for attempt_count in sorted(distribution):
        count = distribution[attempt_count]
        print({
            "attempts": attempt_count,
            "count": count,
            "probability": count / len(values),
        })

    print("worst answers:", worst_answers)

    return {
        "mean_attempts": float(values.mean()),
        "max_attempts": max_attempts,
        "worst_answers": worst_answers,
        "distribution": dict(sorted(distribution.items())),
    }


In [5]:
# ----------------------------
# Data and experiment settings
# ----------------------------

ANSWERS_URL = (
    "https://raw.githubusercontent.com/"
    "pythonmcpi/wordle-wordlist/main/answerlist.txt"
)
GUESSES_URL = (
    "https://raw.githubusercontent.com/"
    "pythonmcpi/wordle-wordlist/main/wordlist.txt"
)

CACHE_PATH = Path(
    ".wordle_mem_cache/pattern_table_uint8.npy"
)

# At every nonterminal candidate state, compare these lambdas.
LAMBDA_GRID = tuple(
    round(float(value), 2)
    for value in np.arange(0.40, 1.001, 0.05)
)


# ----------------------------
# Run dynamic MEM silently
# ----------------------------

# Internal progress and diagnostic messages are suppressed so that the
# notebook displays only the requested final summary.
with contextlib.redirect_stdout(io.StringIO()):
    answers = load_words_from_url(ANSWERS_URL)
    guesses = load_words_from_url(GUESSES_URL)
    guesses = sorted(set(guesses) | set(answers))

    pattern_table = load_or_build_pattern_table(
        answers=answers,
        guesses=guesses,
        cache_path=CACHE_PATH,
    )

    solver = BellmanDynamicMEM(
        answers=answers,
        guesses=guesses,
        pattern_table=pattern_table,
        lambda_grid=LAMBDA_GRID,
        tie_tolerance=1e-12,
        profile_progress_every=0,
        solve_progress_every=0,
        small_state_pairwise_threshold=15,
    )

    root_state = tuple(range(len(answers)))
    root_decision = solver.solve_state(root_state)
    policy_result = solver.evaluate_full_policy(
        root_state=root_state,
        root_decision=root_decision,
    )


# ----------------------------
# Minimal result output
# ----------------------------

attempt_values = np.array(
    list(policy_result["attempts_by_answer"].values()),
    dtype=np.int32,
)
attempt_distribution = Counter(attempt_values.tolist())
total_answers = len(attempt_values)

print("Algorithm: dynamic MEM")
print(f"Initial lambda: {root_decision.chosen_lambda:.2f}")
print(
    "First word:",
    solver.guesses[root_decision.guess_index],
)
print(f"Mean attempts: {attempt_values.mean():.6f}")

cumulative_count = 0
table_rows = []

for attempts in sorted(attempt_distribution):
    answer_count = attempt_distribution[attempts]
    cumulative_count += answer_count

    answer_ratio = 100.0 * answer_count / total_answers
    cumulative_ratio = 100.0 * cumulative_count / total_answers

    table_rows.append(
        "<tr>"
        f"<td>{attempts}</td>"
        f"<td>{answer_count:,}</td>"
        f"<td>{answer_ratio:.2f}%</td>"
        f"<td>{cumulative_ratio:.2f}%</td>"
        "</tr>"
    )

table_html = f"""
<style>
.dynamic-mem-table {{
    border-collapse: collapse;
    margin-top: 10px;
}}
.dynamic-mem-table th,
.dynamic-mem-table td {{
    border: none !important;
    padding: 4px 14px;
    text-align: right;
}}
.dynamic-mem-table th:first-child,
.dynamic-mem-table td:first-child {{
    text-align: center;
}}
</style>
<table class="dynamic-mem-table">
    <thead>
        <tr>
            <th>Attempts</th>
            <th>Answers</th>
            <th>Ratio</th>
            <th>Cumulative ratio</th>
        </tr>
    </thead>
    <tbody>
        {''.join(table_rows)}
    </tbody>
</table>
"""

display(HTML(table_html))


Algorithm: dynamic MEM
Initial lambda: 0.70
First word: reast
Mean attempts: 3.429190


Attempts,Answers,Ratio,Cumulative ratio
2,68,2.94%,2.94%
3,"1,233",53.40%,56.34%
4,957,41.45%,97.79%
5,51,2.21%,100.00%
